# Supervised Classification & Model Tuning — Customer Churn

**Internship deliverable | Classification | Reproducible Scikit-learn workflow**

This notebook compares four distinct classification algorithms, tunes hyperparameters with `GridSearchCV` and stratified K-fold cross-validation, evaluates the selected model, plots ROC curves and a confusion matrix, and saves the final fitted pipeline as a `.joblib` file.

**Dataset:** `customer-churn-training.csv`  
**Target:** `churned` (`1` = churned, `0` = not churned)  
**Identifier excluded from predictors:** `customer_id`

> **Important data-quality note:** The supplied training CSV previously reviewed contains only 12 rows. This is far too small for reliable generalization estimates. The notebook therefore adapts the cross-validation fold count to the minority class and explicitly labels results as educational / highly uncertain. Do not claim the model is production-ready. For stronger evidence, rerun with a larger, authorized dataset.

In [ ]:
# 1. Imports and reproducibility
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 1. Load the dataset

In Google Colab, run the cell below and upload `customer-churn-training.csv` when prompted. In a local Jupyter environment, put the CSV beside this notebook.

In [ ]:
# Colab upload fallback; no upload needed if the CSV already exists in the working directory.
DATA_PATH = "customer-churn-training.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Please upload customer-churn-training.csv.")
        DATA_PATH = csv_files[0]
    except ImportError:
        raise FileNotFoundError(
            "Could not find customer-churn-training.csv. Place it in this notebook's folder."
        )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head())
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(dropna=False)
}))
print("Duplicate rows:", int(df.duplicated().sum()))

## 2. Define target and inspect class balance

The identifier is excluded so the model cannot memorize customer IDs. The target column is never included as a predictor.

In [ ]:
TARGET = "churned"
IDENTIFIER_COLUMNS = ["customer_id"]

if TARGET not in df.columns:
    raise ValueError(f"Expected target column '{TARGET}'. Found: {list(df.columns)}")

df = df.dropna(subset=[TARGET]).copy()
y = df[TARGET]
if y.nunique() != 2:
    raise ValueError(f"This notebook expects a binary target; found labels: {sorted(y.unique().tolist())}")
# Normalize binary target to integers when labels are 0/1.
if set(pd.Series(y).dropna().unique()).issubset({0, 1, 0.0, 1.0}):
    y = y.astype(int)
else:
    raise ValueError("Target 'churned' must use binary labels 0 and 1.")

drop_cols = [c for c in IDENTIFIER_COLUMNS if c in df.columns] + [TARGET]
X = df.drop(columns=drop_cols).copy()

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Target distribution:")
display(y.value_counts().rename(index={0: "Not churned (0)", 1: "Churned (1)"}).to_frame("count"))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Predictor count:", X.shape[1])

## 3. Train/test split and preprocessing pipeline

The held-out test set is separated before model fitting. Imputation, scaling, and categorical encoding are fitted inside each training fold via a `Pipeline`, reducing data leakage. For a very small dataset, test scores will be unstable.

In [ ]:
class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError("At least two rows per class are needed for a stratified train/test split.")

# A 25% test split is a compromise for this tiny educational dataset.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
], remainder="drop")

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Training class counts:")
display(y_train.value_counts().sort_index().to_frame("count"))
print("Test class counts:")
display(y_test.value_counts().sort_index().to_frame("count"))

## 4. Train and tune four distinct model architectures

Models compared:
1. Logistic Regression — linear baseline
2. Random Forest — bagged decision trees
3. Extra Trees — randomized tree ensemble
4. Gradient Boosting — sequential boosting ensemble

`GridSearchCV` uses stratified K-fold cross-validation. The fold count is automatically limited by the smallest class in the training split. The primary tuning metric is F1-score to balance precision and recall for churn identification.

In [ ]:
min_train_class = int(y_train.value_counts().min())
N_SPLITS = min(3, min_train_class)
if N_SPLITS < 2:
    raise ValueError("Not enough examples per class in the training split for stratified cross-validation.")

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

model_specs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
        {"classifier__C": [0.1, 1.0, 10.0]}
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 3],
            "classifier__min_samples_leaf": [1, 2]
        }
    ),
    "Extra Trees": (
        ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 3],
            "classifier__min_samples_leaf": [1, 2]
        }
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "classifier__n_estimators": [50, 100],
            "classifier__learning_rate": [0.03, 0.1],
            "classifier__max_depth": [1, 2]
        }
    )
}

searches = {}
comparison_rows = []

for model_name, (estimator, param_grid) in model_specs.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        refit=True,
        error_score="raise",
        return_train_score=True
    )
    search.fit(X_train, y_train)
    searches[model_name] = search

    fitted = search.best_estimator_
    pred = fitted.predict(X_test)
    proba = fitted.predict_proba(X_test)[:, 1] if hasattr(fitted, "predict_proba") else None

    # ROC-AUC is undefined if the held-out test set contains only one class.
    test_auc = roc_auc_score(y_test, proba) if proba is not None and y_test.nunique() == 2 else np.nan
    comparison_rows.append({
        "Model": model_name,
        "Best CV F1": search.best_score_,
        "Test Accuracy": accuracy_score(y_test, pred),
        "Test Precision": precision_score(y_test, pred, zero_division=0),
        "Test Recall": recall_score(y_test, pred, zero_division=0),
        "Test F1": f1_score(y_test, pred, zero_division=0),
        "Test ROC-AUC": test_auc,
        "Best Parameters": str(search.best_params_)
    })
    print(f"Finished: {model_name} | best CV F1 = {search.best_score_:.3f}")

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ["Best CV F1", "Test F1"], ascending=False
).reset_index(drop=True)
display(comparison_df.round(3))

## 5. Select the champion model

The champion is selected by **mean cross-validation F1-score** rather than by the held-out test score. The test set is kept for a final, one-time evaluation. With only 12 records, small differences should not be treated as statistically meaningful.

In [ ]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_

print("Champion model:", champion_name)
print("Best mean CV F1-score:", round(champion_search.best_score_, 4))
print("Selected hyperparameters:")
display(pd.Series(champion_search.best_params_, name="value").to_frame())

y_pred = champion_model.predict(X_test)
y_proba = champion_model.predict_proba(X_test)[:, 1]

print("Held-out test classification report:")
print(classification_report(y_test, y_pred, labels=[0, 1],
                            target_names=["Not churned (0)", "Churned (1)"],
                            zero_division=0))
print("Confusion matrix (rows = actual, columns = predicted):")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=[0, 1],
    display_labels=["Not churned", "Churned"],
    cmap="Blues", values_format="d"
)
plt.title(f"Champion Confusion Matrix — {champion_name}")
plt.tight_layout()
plt.show()

if y_test.nunique() == 2:
    print("Held-out test ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
else:
    print("ROC-AUC not reported: the held-out test set contains only one class.")

## 6. ROC curves

The ROC curve visualizes the true-positive rate against the false-positive rate at different thresholds. Test-set ROC-AUC is included only when both classes appear in the held-out test set.

In [ ]:
plt.figure(figsize=(8, 6))
plotted = 0
for model_name, search in searches.items():
    model = search.best_estimator_
    probabilities = model.predict_proba(X_test)[:, 1]
    if y_test.nunique() == 2:
        RocCurveDisplay.from_predictions(
            y_test, probabilities, name=model_name, ax=plt.gca()
        )
        plotted += 1
if plotted:
    plt.plot([0, 1], [0, 1], linestyle="--", label="Chance baseline")
    plt.title("ROC Curves — Tuned Models (Held-out Test Set)")
    plt.tight_layout()
    plt.legend()
    plt.show()
else:
    print("ROC curves cannot be drawn because the test set contains only one target class.")

## 7. Hyperparameter tuning summary

This table reports the best F1-score found by cross-validation for each model and the hyperparameters selected by grid search.

In [ ]:
tuning_summary = pd.DataFrame([
    {
        "Model": name,
        "CV folds": N_SPLITS,
        "Best CV F1": search.best_score_,
        "Parameter combinations tested": len(search.cv_results_["params"]),
        "Best parameters": str(search.best_params_)
    }
    for name, search in searches.items()
]).sort_values("Best CV F1", ascending=False).reset_index(drop=True)
display(tuning_summary.round(4))

## 8. Save the champion model and comparison results

The serialized artifact includes preprocessing and the classifier, so new data can pass through the same transformations. Do not publish row-level customer data. The notebook, plots, comparison table, and model artifact are the expected proof items.

In [ ]:
MODEL_PATH = "champion_churn_model.joblib"
TABLE_PATH = "model_comparison.csv"

joblib.dump(champion_model, MODEL_PATH)
comparison_df.to_csv(TABLE_PATH, index=False)

print(f"Saved champion pipeline: {MODEL_PATH}")
print(f"Saved comparison table: {TABLE_PATH}")
print("Model file size (bytes):", os.path.getsize(MODEL_PATH))
print("Saved files are in the notebook working directory.")

## 9. Final conclusion and limitations

**Workflow completed:** four model architectures; preprocessing pipeline; stratified K-fold cross-validation; GridSearchCV hyperparameter tuning; model comparison table; precision, recall, F1, accuracy and ROC-AUC where defined; confusion matrix; ROC curves; champion selection; serialized `.joblib` model.

**Interpretation:** Select the champion based on the displayed cross-validation results, but do not overstate performance. The supplied file contains only 12 records, and the observed plan-type/target pattern may be unusually strong. A very small test set can produce deceptively high or low scores. Reliable conclusions require a larger, representative, authorized dataset and confirmation that predictors are available before the churn outcome.

**Reproducibility:** Keep the notebook and CSV together in Colab, run all cells from top to bottom, confirm that the comparison table and plots appear, then save the executed notebook. Share the notebook using a public Viewer link only if your internship platform permits it; avoid making the raw customer CSV publicly accessible.